# PixPay — Auditoria de Qualidade dos Dados

**Objetivo:** Antes de qualquer análise causal, mapear exatamente o que está sujo no dataset `clientes_pixpay.csv`.  
**Escopo:** Valores ausentes, duplicatas e inconsistências de formato em colunas categóricas/chave.  
**Data:** 22/05/2026

In [ ]:
import pandas as pd
import numpy as np
import re

# Carrega sem conversão automática para preservar os valores brutos
df = pd.read_csv('../data/clientes_pixpay.csv', dtype=str)

print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')

## 1. Valores ausentes por coluna

In [ ]:
def audit_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Conta NaN e strings vazias — ambos são 'ausente' no contexto deste dataset."""
    total = len(df)
    rows = []
    for col in df.columns:
        nan_count = df[col].isna().sum()
        # Strings vazias ou só espaços também contam como ausentes
        empty_str = df[col].dropna().str.strip().eq('').sum()
        total_missing = nan_count + empty_str
        rows.append({
            'coluna': col,
            'nan': nan_count,
            'string_vazia': empty_str,
            'total_ausente': total_missing,
            'pct_ausente': round(total_missing / total * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('total_ausente', ascending=False)

missing_df = audit_missing(df)
print('=== VALORES AUSENTES ===')
display(missing_df[missing_df['total_ausente'] > 0])

## 2. Duplicatas

In [ ]:
def audit_duplicatas(df: pd.DataFrame) -> dict:
    """
    Verifica duplicatas em três níveis:
    - Linhas completamente idênticas
    - cliente_id repetido (deve ser chave primária)
    - CPF repetido (pode indicar mesma pessoa com contas distintas)
    """
    resultado = {}

    # Linhas completamente idênticas
    linhas_dup = df.duplicated().sum()
    resultado['linhas_identicas'] = linhas_dup

    # cliente_id duplicado
    dup_id = df['cliente_id'].duplicated(keep=False)
    resultado['cliente_id_duplicado_count'] = dup_id.sum()
    resultado['cliente_id_duplicados'] = df[dup_id][['cliente_id', 'nome', 'cpf', 'data_cadastro']]

    # CPF duplicado (normaliza antes: remove pontuação)
    cpf_norm = df['cpf'].str.replace(r'[.\-]', '', regex=True).str.strip()
    dup_cpf = cpf_norm.duplicated(keep=False)
    resultado['cpf_duplicado_count'] = dup_cpf.sum()
    resultado['cpf_duplicados'] = df[dup_cpf][['cliente_id', 'nome', 'cpf', 'data_cadastro']]

    return resultado

dup = audit_duplicatas(df)

print(f"=== DUPLICATAS ===")
print(f"Linhas completamente idênticas : {dup['linhas_identicas']}")
print(f"Registros com cliente_id repetido: {dup['cliente_id_duplicado_count']}")
print(f"Registros com CPF repetido       : {dup['cpf_duplicado_count']}")

if dup['cliente_id_duplicado_count'] > 0:
    print('\n--- cliente_id duplicados ---')
    display(dup['cliente_id_duplicados'])

if dup['cpf_duplicado_count'] > 0:
    print('\n--- CPFs duplicados (após normalização) ---')
    display(dup['cpf_duplicados'])

## 3. Inconsistências de formato — colunas categóricas e chave

### 3a. Coluna `plano`

In [ ]:
def audit_categorica(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    """Retorna contagem de cada variante bruta, incluindo espaços e capitalização."""
    return (
        df[coluna]
        .value_counts(dropna=False)
        .rename_axis('valor_bruto')
        .reset_index(name='contagem')
    )

print('=== COLUNA: plano ===')
display(audit_categorica(df, 'plano'))

# Valores únicos após normalização para ver quantos planos reais existem
planos_norm = df['plano'].str.strip().str.lower().value_counts(dropna=False)
print('\nApós strip + lower:')
display(planos_norm)

### 3b. Coluna `valor_mensalidade`

In [ ]:
def audit_valor_mensalidade(df: pd.DataFrame) -> dict:
    """
    Classifica cada entrada em padrões de formato para quantificar
    o esforço de limpeza necessário.
    """
    col = df['valor_mensalidade'].fillna('')

    # Padrão: "R$ 19,90" ou "R$ 49,90"
    fmt_rs_virgula = col.str.match(r'^R\$\s*\d+,\d{2}$')
    # Padrão: "19,90" (sem prefixo, vírgula decimal)
    fmt_virgula    = col.str.match(r'^\d+,\d{2}$')
    # Padrão: "49.90" (sem prefixo, ponto decimal)
    fmt_ponto      = col.str.match(r'^\d+\.\d{2}$')
    # Resto
    fmt_outro      = ~(fmt_rs_virgula | fmt_virgula | fmt_ponto) & col.ne('')
    fmt_vazio      = col.eq('')

    return {
        'R$ com vírgula (ex: R$ 19,90)': fmt_rs_virgula.sum(),
        'só número com vírgula (ex: 19,90)': fmt_virgula.sum(),
        'só número com ponto (ex: 49.90)': fmt_ponto.sum(),
        'outro formato': fmt_outro.sum(),
        'vazio/ausente': fmt_vazio.sum(),
    }

print('=== COLUNA: valor_mensalidade ===')
for padrao, qtd in audit_valor_mensalidade(df).items():
    print(f'  {padrao:45s}: {qtd:>6}')

print('\nAmostras brutas únicas:')
display(audit_categorica(df, 'valor_mensalidade'))

### 3c. Coluna `data_cadastro` e `data_cancelamento`

In [ ]:
def classifica_formato_data(series: pd.Series) -> pd.Series:
    """
    Rotula cada valor com o padrão de formato identificado.
    Retorna Series com o rótulo por linha.
    """
    def detecta(v):
        if pd.isna(v) or str(v).strip() == '':
            return 'ausente'
        v = str(v).strip()
        if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
            return 'ISO (AAAA-MM-DD)'
        if re.match(r'^\d{2}/\d{2}/\d{4}$', v):
            return 'BR (DD/MM/AAAA)'
        if re.match(r'^\d{2}-\d{2}-\d{2}$', v):
            return 'abreviado (DD-MM-AA)'
        if re.match(r'^\d{2}-\d{2}-\d{4}$', v):
            return 'hifenizado (DD-MM-AAAA)'
        return f'desconhecido: {v}'
    return series.apply(detecta)

for col_data in ['data_cadastro', 'data_cancelamento']:
    rotulos = classifica_formato_data(df[col_data])
    print(f'=== COLUNA: {col_data} ===')
    display(rotulos.value_counts(dropna=False).rename_axis('formato').reset_index(name='contagem'))
    print()

### 3d. Coluna `abriu_chamado` e `churn_30d`

In [ ]:
for col_bool in ['abriu_chamado', 'churn_30d']:
    print(f'=== COLUNA: {col_bool} ===')
    display(audit_categorica(df, col_bool))
    print()

### 3e. Coluna `cpf` — formatos

In [ ]:
def classifica_formato_cpf(series: pd.Series) -> pd.Series:
    def detecta(v):
        if pd.isna(v) or str(v).strip() == '':
            return 'ausente'
        v = str(v).strip()
        if re.match(r'^\d{3}\.\d{3}\.\d{3}-\d{2}$', v):
            return 'formatado (XXX.XXX.XXX-XX)'
        if re.match(r'^\d{11}$', v):
            return 'somente dígitos (11 chars)'
        if re.match(r'^\d{10}$', v):
            # Provavelmente com zero à esquerda ausente
            return 'somente dígitos (10 chars — zero à esquerda?)'
        return f'outro: {v}'
    return series.apply(detecta)

print('=== COLUNA: cpf ===')
rotulos_cpf = classifica_formato_cpf(df['cpf'])
display(rotulos_cpf.value_counts(dropna=False).rename_axis('formato').reset_index(name='contagem'))

### 3f. Coluna `estado` — UFs inválidas

In [ ]:
UFS_VALIDAS = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS',
    'MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC',
    'SP','SE','TO'
}

estados = df['estado'].str.strip().str.upper()
invalidos = estados[~estados.isin(UFS_VALIDAS) & estados.notna()]

print(f'=== COLUNA: estado ===')
print(f'UFs inválidas ou com formatação estranha: {len(invalidos)}')
if len(invalidos) > 0:
    display(invalidos.value_counts())

## 4. Outliers óbvios em colunas numéricas

In [ ]:
def parse_valor_mensalidade(v):
    """Converte qualquer variante de formato para float."""
    if pd.isna(v):
        return np.nan
    v = str(v).strip()
    v = v.replace('R$', '').strip()
    v = v.replace(',', '.')
    try:
        return float(v)
    except ValueError:
        return np.nan

df_num = df.copy()
df_num['valor_mensalidade_num'] = df['valor_mensalidade'].apply(parse_valor_mensalidade)
df_num['transacoes_30d_num']    = pd.to_numeric(df['transacoes_30d'], errors='coerce')
df_num['ticket_medio_30d_num']  = pd.to_numeric(df['ticket_medio_30d'], errors='coerce')
df_num['notificacoes_7d_num']   = pd.to_numeric(df['notificacoes_7d'], errors='coerce')

colunas_numericas = [
    'valor_mensalidade_num', 'transacoes_30d_num',
    'ticket_medio_30d_num', 'notificacoes_7d_num'
]

print('=== ESTATÍSTICAS DESCRITIVAS (pós-parse) ===')
display(df_num[colunas_numericas].describe().round(2))

# Outliers via IQR
print('\n=== OUTLIERS (> Q3 + 3×IQR) ===')
for col in colunas_numericas:
    s = df_num[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    limite = q3 + 3 * iqr
    extremos = df_num[df_num[col] > limite][['cliente_id', col]]
    print(f'{col}: {len(extremos)} outlier(s) acima de {limite:.2f}')
    if len(extremos) > 0 and len(extremos) <= 10:
        display(extremos)

## 5. Resumo executivo da auditoria

In [ ]:
print('=' * 60)
print('RESUMO DA AUDITORIA — clientes_pixpay.csv')
print('=' * 60)
total = len(df)
print(f'Total de registros : {total:>7}')
print()
print('AUSENTES')
for _, row in missing_df[missing_df['total_ausente'] > 0].iterrows():
    print(f"  {row['coluna']:35s}: {row['total_ausente']:>5} ({row['pct_ausente']}%)")
print()
print('DUPLICATAS')
print(f"  Linhas idênticas              : {dup['linhas_identicas']}")
print(f"  cliente_id repetido           : {dup['cliente_id_duplicado_count']}")
print(f"  CPF repetido (normalizado)    : {dup['cpf_duplicado_count']}")
print()
print('INCONSISTÊNCIAS CONHECIDAS')
print('  plano            : capitalização variada (ex: PREMIUM vs Premium vs Básico)')
print('  valor_mensalidade: 3+ formatos (R$ X,Y | X,Y | X.Y)')
print('  data_cadastro    : 3+ formatos de data')
print('  abriu_chamado    : mix booleano Python + string "não"')
print('  cpf              : formatado vs somente dígitos')
print()
print('PRÓXIMO PASSO: limpeza orientada por hipóteses de churn,')
print('confirmando cada descarte com o gestor de dados.')
print('=' * 60)